# 第111章 交叉验证策略

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 26 / 34 步：调优、比较、解释并保存模型**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** PCA降维与可视化  →  **本章任务：** 交叉验证策略  →  **下一步：** GridSearch与随机搜索
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：拿到一份数据后，真正的难题往往不是“把模型训练出来”，而是“这个模型换一批数据还能不能这么准”。交叉验证把数据切成几折轮流当验证集，让一次评分变成对多组数据的多次检验，帮你在上真实场景之前就先摸清模型的稳定性，避免被某一批样本的偶然结果带偏。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交叉验证策略」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「交叉验证策略」的关键输出指标。
- **迁移**：能把「交叉验证策略」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：交叉验证的常见做法是把数据随机切几折，但遇到时间、客户这类有次序的依赖，随机切会“偷看未来”。交叉验证策略的核心是：用哪种切法更接近真实使用场景。分类要保持类别比例、同一客户不能跨折、未来数据不能进入过去训练——这些才是它真正的难点。


- CV均值：\(K^{-1}\sum_k score_k\)
- 分类通常保持类别比例
- 同一客户不能跨训练验证折
- 未来数据不能进入过去训练（打个比方：时间数据不能像抽签那样打乱——不能用“后天”的数据解释“今天”，就像不能拿已经发生的结果去倒推当时；要用“过去”学，再用“未来”验。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | 参见本节示例 | 先明确样本、特征、目标和验证方式，再训练模型。 | 时间序列使用随机 KFold |
| 模型、公式与诊断 | `s.mean()`、`s.std()`、`s.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 同一用户记录跨折 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-111 -->
### 数学推导｜交叉验证汇总泛化波动

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把样本分成 $K$ 个互斥验证折。** 第 $k$ 次用其余折训练，在第 $k$ 折得到分数 $s_k$。

**第 2 步｜平均各折表现。** $\bar s=\sum_ks_k/K$ 近似描述该训练流程在不同样本划分下的表现。

**第 3 步｜报告划分敏感性。** 标准差衡量折间波动；若只想描述均值估计的不确定度，可另算

$$
SE(\bar s)\approx\frac{SD(s)}{\sqrt K}
$$

但各折训练集高度重叠，并不严格独立，所以这个标准误只能谨慎参考。

**把上面的关系收束为本章计算式：**

$$
\bar{s}=\frac{1}{K}\sum_{k=1}^{K}s_k,\qquad SD(s)=\sqrt{\frac{1}{K-1}\sum_k(s_k-\bar{s})^2}
$$

**符号解释：** $s_k$ 是第 $k$ 折验证分数。

**代码对应：** 同时报告均值和标准差，并选择与时间、分组或类别结构匹配的切分器。

**使用边界：** 交叉验证折并非完全独立；时间数据不能随机打乱未来与过去。


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：C.1 调整验证折数，观察评分变化

示例 1 用 3、5、10 三种折数跑出了均值。现在保持数据和模型不变（复用 `X`、`y`、`model`），只把验证折数 `n_splits` 改成自己的取值（例如 4 或 6）重新跑一遍，记录均值与标准差的变化，并思考：**折数变多时，每次用作验证的样本少了，评分稳定性往往会怎样变？**

> 提示：若当前环境还没有 `X`、`y`、`model`，先运行示例 1 的代码；折数取 2~10 之间的整数。


In [ ]:
try:
    # 请在下方填写代码：把下方 my_folds_value 替换成自己的折数，例如 4 或 6
    import numpy as np
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    my_folds = 4  # ← 请替换成自己的折数（如 3、5、10）
    cv = StratifiedKFold(n_splits=my_folds, shuffle=True, random_state=100)
    s = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(
        "折数：",
        my_folds,
        "CV均值：",
        round(s.mean(), 4),
        "标准差：",
        round(s.std(), 4),
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
for folds in [3, 5, 10]:
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=100)
    s = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(
        folds,
        "mean",
        round(s.mean(), 4),
        "std",
        round(s.std(), 4),
        "scores",
        s.round(3),
    )


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
_demo_model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(_demo_model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, _demo_model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = _demo_model.predict(X_changed)
print("原始前2个预测：", np.round(_demo_model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(changed_prediction[:2] - _demo_model.predict(X[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 时间序列使用随机 KFold
- 同一用户记录跨折
- 只报告 CV 最佳折
- 交叉验证后仍用相同数据声称独立测试


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 111.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 111.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 111.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

为独立同分布、分组数据和时间序列选择正确交叉验证策略。


### 你已经掌握

- 使用 KFold 与 StratifiedKFold
- 使用 GroupKFold 防止实体泄漏
- 使用 TimeSeriesSplit 保持时间顺序
- 报告均值与标准差


### 需要注意

- 时间序列使用随机 KFold
- 同一用户记录跨折
- 只报告 CV 最佳折
- 交叉验证后仍用相同数据声称独立测试


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 恢复示例 1 的数据与模型（后面的错误演示临时覆盖了它们）
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# 参考实现：把折数改成 4，再与示例中的 3、5、10 对比
my_folds = 4
cv = StratifiedKFold(n_splits=my_folds, shuffle=True, random_state=100)
s = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
print(
    "折数：",
    my_folds,
    "CV均值：",
    round(s.mean(), 4),
    "标准差：",
    round(s.std(), 4),
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

ts = TimeSeriesSplit(n_splits=4)
practice_splits = [
    (train[-1], test[0]) for train, test in ts.split(np.arange(100))
]
print(practice_splits)
